# National Job Market Predictor

This notebook uses the BLS/OEWS national occupation files downloaded so far to build an occupation-level forecasting model. The current data covers 2010-2015, so the model is useful as a prototype and backtesting template, not yet as a reliable four-year national forecast.

**Model choice:** Random Forest regression is used as the main model. With only annual observations, an RNN/LSTM would be poorly supported because sequence depth is very short. Random Forest can use the available cross-sectional occupation data, nonlinear relationships, and lagged features without pretending there is enough time-series depth for deep learning.

## Report Summary

### Data Used

The notebook reads national BLS/OEWS occupation-level CSV files from both `converted_csv/` and the original `oesm*nat/` folders. The current downloaded national files cover 2010-2015.

### Preprocessing

- Standardizes schema changes such as `GROUP` versus `OCC_GROUP`.
- Extracts the year from each file name.
- Converts BLS numeric strings such as `"127,097,160"` into numbers.
- Treats suppression markers such as `*`, `**`, `#`, and blanks as missing values.
- Keeps detailed occupation rows and removes aggregate total/major/minor/broad rows.
- Builds a panel dataset with one row per `year + occupation`.
- Creates lagged one-year-ahead targets for employment and annual mean wage.

### Modeling

The model predicts next-year `TOT_EMP` and `A_MEAN` using the current year's occupation features. It uses a `RandomForestRegressor` inside a preprocessing pipeline with categorical encoding for occupation code/title and numeric scaling/imputation.

### Validation

The notebook uses a time-aware split: it trains on transitions up to 2014 and tests on the 2014-to-2015 transition. This avoids randomly mixing future years into training.

### Limitation

With only 2010-2015, this is a proof of concept. For the original project goal, download 2016-2025 before treating the results as meaningful. Four-year forecasts from only six yearly snapshots should be presented as experimental.

In [1]:
from pathlib import Path
import re

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path.cwd()
CONVERTED_ROOT = PROJECT_ROOT / "converted_csv"

In [2]:
def extract_year(path: Path) -> int:
    match = re.search(r"M(20\d{2})", path.name)
    if not match:
        match = re.search(r"oesm(\d{2})", str(path))
    if not match:
        raise ValueError(f"Could not determine year from {path}")
    year_text = match.group(1)
    return int(year_text if len(year_text) == 4 else f"20{year_text}")


def discover_national_files(project_root: Path) -> list[Path]:
    converted = list((project_root / "converted_csv").glob("oesm*nat/national_M*_dl__national_dl.csv"))
    original = list(project_root.glob("oesm*nat/national_M*_dl.csv"))

    by_year: dict[int, Path] = {}
    for path in sorted(converted + original):
        year = extract_year(path)
        # Prefer converted files, but fall back to original CSVs where needed.
        if year not in by_year or "converted_csv" in path.parts:
            by_year[year] = path
    return [by_year[year] for year in sorted(by_year)]


national_files = discover_national_files(PROJECT_ROOT)
print(f"Found {len(national_files)} national occupation files")
for path in national_files:
    print(extract_year(path), path.relative_to(PROJECT_ROOT))

Found 6 national occupation files
2010 converted_csv/oesm10nat/national_M2010_dl__national_dl.csv
2011 oesm11nat/national_M2011_dl.csv
2012 converted_csv/oesm12nat/national_M2012_dl__national_dl.csv
2013 converted_csv/oesm13nat/national_M2013_dl__national_dl.csv
2014 converted_csv/oesm14nat/national_M2014_dl__national_dl.csv
2015 converted_csv/oesm15nat/national_M2015_dl__national_dl.csv


In [3]:
SUPPRESSED_VALUES = {"", "*", "**", "#", "~"}
NUMERIC_COLUMNS = [
    "TOT_EMP",
    "EMP_PRSE",
    "H_MEAN",
    "A_MEAN",
    "MEAN_PRSE",
    "H_PCT10",
    "H_PCT25",
    "H_MEDIAN",
    "H_PCT75",
    "H_PCT90",
    "A_PCT10",
    "A_PCT25",
    "A_MEDIAN",
    "A_PCT75",
    "A_PCT90",
]


def clean_numeric(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype("string")
        .str.strip()
        .str.replace(",", "", regex=False)
        .replace(list(SUPPRESSED_VALUES), pd.NA)
    )
    return pd.to_numeric(cleaned, errors="coerce")


def load_national_file(path: Path) -> pd.DataFrame:
    data = pd.read_csv(path, dtype=str)
    data.columns = [column.strip().upper() for column in data.columns]
    data = data.rename(columns={"OCC_GROUP": "GROUP"})
    data["YEAR"] = extract_year(path)
    data["SOURCE_FILE"] = str(path.relative_to(PROJECT_ROOT))

    for column in NUMERIC_COLUMNS:
        if column in data.columns:
            data[column] = clean_numeric(data[column])

    data["GROUP"] = data.get("GROUP", "").fillna("").astype(str).str.lower().str.strip()
    return data


raw_panel = pd.concat([load_national_file(path) for path in national_files], ignore_index=True)
print(raw_panel.shape)
raw_panel.head()

(7214, 22)


,OCC_CODE,OCC_TITLE,GROUP,TOT_EMP,EMP_PRSE,H_MEAN,A_MEAN,MEAN_PRSE,H_PCT10,H_PCT25,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY,YEAR,SOURCE_FILE
0,00-0000,All Occupations,total,127097160,0.1,21.35,44410,0.1,8.51,10.65,16.27,26.08,39.97,17690,22150,33840,54250,83140,NaN,NaN,2010,converted_csv/oesm10nat/national_M2010_dl__nat...
1,11-0000,Management Occupations,major,6022860,0.2,50.69,105440,0.1,21.57,30.65,43.96,62.97,<NA>,44860,63760,91440,130980,<NA>,NaN,NaN,2010,converted_csv/oesm10nat/national_M2010_dl__nat...
2,11-1011,Chief Executives,,273500,0.5,83.34,173350,0.3,36.14,51.92,79.37,<NA>,<NA>,75160,107990,165080,<NA>,<NA>,NaN,NaN,2010,converted_csv/oesm10nat/national_M2010_dl__nat...
3,11-1021,General and Operations Managers,,1708080,0.3,54.38,113100,0.2,22.73,31.39,45.38,68.28,<NA>,47280,65290,94400,142030,<NA>,NaN,NaN,2010,converted_csv/oesm10nat/national_M2010_dl__nat...
4,11-1031,Legislators,,65710,1.3,<NA>,38470,1.2,<NA>,<NA>,<NA>,<NA>,<NA>,15790,16790,19260,54170,84320,True,NaN,2010,converted_csv/oesm10nat/national_M2010_dl__nat...


In [4]:
# Use detailed occupations only. Older files mark details with a blank group; newer files use 'detailed'.
detail_groups = {"", "detailed"}
panel = raw_panel[raw_panel["GROUP"].isin(detail_groups)].copy()
panel = panel[panel["OCC_CODE"].notna() & (panel["OCC_CODE"] != "00-0000")]
panel = panel.sort_values(["OCC_CODE", "YEAR"]).reset_index(drop=True)

# Add stable occupation-code features. These help the model generalize across related occupations.
panel["OCC_MAJOR"] = panel["OCC_CODE"].str.slice(0, 2)
panel["OCC_MINOR"] = panel["OCC_CODE"].str.slice(0, 5)
panel["LOG_TOT_EMP"] = np.log1p(panel["TOT_EMP"])
panel["LOG_A_MEAN"] = np.log1p(panel["A_MEAN"])
panel["EMPLOYMENT_SHARE"] = panel["TOT_EMP"] / panel.groupby("YEAR")["TOT_EMP"].transform("sum")

print("Years:", sorted(panel["YEAR"].unique()))
print("Detailed occupation rows:", panel.shape[0])
print("Unique occupations:", panel["OCC_CODE"].nunique())
panel.head()

Years: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015)]
Detailed occupation rows: 4872
Unique occupations: 840


,OCC_CODE,OCC_TITLE,GROUP,TOT_EMP,EMP_PRSE,H_MEAN,A_MEAN,MEAN_PRSE,H_PCT10,H_PCT25,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY,YEAR,SOURCE_FILE,OCC_MAJOR,OCC_MINOR,LOG_TOT_EMP,LOG_A_MEAN,EMPLOYMENT_SHARE
0,11-1011,Chief Executives,,273500,0.5,83.34,173350,0.3,36.14,51.92,79.37,<NA>,<NA>,75160,107990,165080,<NA>,<NA>,NaN,NaN,2010,converted_csv/oesm10nat/national_M2010_dl__nat...,11,11-10,12.519061,12.063074,0.002152
1,11-1011,Chief Executives,,267370,0.5,84.88,176550,0.4,36.47,52.56,80.25,<NA>,<NA>,75860,109320,166910,<NA>,<NA>,NaN,NaN,2011,oesm11nat/national_M2011_dl.csv,11,11-10,12.496392,12.081365,0.002084
2,11-1011,Chief Executives,detailed,255940,0.6,85.02,176840,0.3,36.65,52.86,80.84,<NA>,<NA>,76220,109940,168140,<NA>,<NA>,NaN,NaN,2012,converted_csv/oesm12nat/national_M2012_dl__nat...,11,11-10,12.452702,12.083006,0.001964
3,11-1011,Chief Executives,detailed,248760,0.6,85.77,178400,0.3,36.07,53.18,82.5,<NA>,<NA>,75030,110610,171610,<NA>,<NA>,NaN,NaN,2013,converted_csv/oesm13nat/national_M2013_dl__nat...,11,11-10,12.424248,12.091789,0.001876
4,11-1011,Chief Executives,detailed,246240,0.8,86.88,180700,0.4,34.97,53.25,83.33,<NA>,<NA>,72750,110760,173320,<NA>,<NA>,NaN,NaN,2014,converted_csv/oesm14nat/national_M2014_dl__nat...,11,11-10,12.414066,12.104599,0.001822


In [5]:
# Create one-year-ahead targets by occupation.
panel["NEXT_YEAR"] = panel.groupby("OCC_CODE")["YEAR"].shift(-1)
panel["NEXT_TOT_EMP"] = panel.groupby("OCC_CODE")["TOT_EMP"].shift(-1)
panel["NEXT_A_MEAN"] = panel.groupby("OCC_CODE")["A_MEAN"].shift(-1)
panel["TARGET_LOG_TOT_EMP"] = np.log1p(panel["NEXT_TOT_EMP"])
panel["TARGET_LOG_A_MEAN"] = np.log1p(panel["NEXT_A_MEAN"])

model_data = panel[
    panel["NEXT_YEAR"].eq(panel["YEAR"] + 1)
    & panel["TARGET_LOG_TOT_EMP"].notna()
    & panel["TARGET_LOG_A_MEAN"].notna()
].copy()

print(model_data[["YEAR", "NEXT_YEAR"]].drop_duplicates().sort_values(["YEAR", "NEXT_YEAR"]))
print("Model rows:", model_data.shape[0])

   YEAR  NEXT_YEAR
0  2010     2011.0
1  2011     2012.0
2  2012     2013.0
3  2013     2014.0
4  2014     2015.0
Model rows: 4012


In [7]:
FEATURE_COLUMNS = [
    "YEAR",
    "OCC_CODE",
    "OCC_TITLE",
    "OCC_MAJOR",
    "OCC_MINOR",
    "TOT_EMP",
    "EMP_PRSE",
    "H_MEAN",
    "A_MEAN",
    "MEAN_PRSE",
    "H_PCT10",
    "H_PCT25",
    "H_MEDIAN",
    "H_PCT75",
    "H_PCT90",
    "A_PCT10",
    "A_PCT25",
    "A_MEDIAN",
    "A_PCT75",
    "A_PCT90",
    "LOG_TOT_EMP",
    "LOG_A_MEAN",
    "EMPLOYMENT_SHARE",
]
TARGET_COLUMNS = ["TARGET_LOG_TOT_EMP", "TARGET_LOG_A_MEAN"]

available_features = [column for column in FEATURE_COLUMNS if column in model_data.columns]
categorical_features = ["OCC_CODE", "OCC_TITLE", "OCC_MAJOR", "OCC_MINOR"]
numeric_features = [column for column in available_features if column not in categorical_features]

last_target_year = int(model_data["NEXT_YEAR"].max())
test_feature_year = last_target_year - 1
train_data = model_data[model_data["YEAR"] < test_feature_year].copy()
test_data = model_data[model_data["YEAR"] == test_feature_year].copy()

print(f"Training transitions: {train_data['YEAR'].min()}->{int(train_data['NEXT_YEAR'].min())} through {train_data['YEAR'].max()}->{int(train_data['NEXT_YEAR'].max())}")
print(f"Testing transition: {test_feature_year}->{last_target_year}")
print("Train rows:", train_data.shape[0], "Test rows:", test_data.shape[0])

Training transitions: 2010->2011 through 2013->2014
Testing transition: 2014->2015
Train rows: 3196 Test rows: 816


In [8]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore", min_frequency=2)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "forest",
            RandomForestRegressor(
                n_estimators=500,
                min_samples_leaf=3,
                random_state=42,
                n_jobs=1,
            ),
        ),
    ]
)

model.fit(train_data[available_features], train_data[TARGET_COLUMNS])
predicted_log = model.predict(test_data[available_features])
predictions = pd.DataFrame(
    {
        "YEAR": test_data["YEAR"].to_numpy(),
        "TARGET_YEAR": test_data["NEXT_YEAR"].to_numpy().astype(int),
        "OCC_CODE": test_data["OCC_CODE"].to_numpy(),
        "OCC_TITLE": test_data["OCC_TITLE"].to_numpy(),
        "ACTUAL_TOT_EMP": test_data["NEXT_TOT_EMP"].to_numpy(),
        "PREDICTED_TOT_EMP": np.expm1(predicted_log[:, 0]),
        "ACTUAL_A_MEAN": test_data["NEXT_A_MEAN"].to_numpy(),
        "PREDICTED_A_MEAN": np.expm1(predicted_log[:, 1]),
    }
)
predictions.head()

,YEAR,TARGET_YEAR,OCC_CODE,OCC_TITLE,ACTUAL_TOT_EMP,PREDICTED_TOT_EMP,ACTUAL_A_MEAN,PREDICTED_A_MEAN
0,2014,2015,11-1011,Chief Executives,238940,2.478984e+05,185850,153110.100584
1,2014,2015,11-1021,General and Operations Managers,2145140,2.275201e+06,119460,80453.152021
2,2014,2015,11-1031,Legislators,55820,5.481572e+04,42530,41462.666577
3,2014,2015,11-2011,Advertising and Promotions Managers,29340,2.987920e+04,113610,109073.942375
4,2014,2015,11-2021,Marketing Managers,192890,1.801996e+05,140660,136276.484727


In [9]:
def regression_report(actual: pd.Series, predicted: pd.Series) -> dict[str, float]:
    actual = pd.Series(actual, dtype="float64")
    predicted = pd.Series(predicted, dtype="float64")
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "WAPE": (actual.sub(predicted).abs().sum() / actual.abs().sum()),
        "R2": r2_score(actual, predicted),
    }


metrics = pd.DataFrame(
    [
        {"target": "TOT_EMP", **regression_report(predictions["ACTUAL_TOT_EMP"], predictions["PREDICTED_TOT_EMP"])},
        {"target": "A_MEAN", **regression_report(predictions["ACTUAL_A_MEAN"], predictions["PREDICTED_A_MEAN"])},
    ]
)
metrics

,target,MAE,RMSE,WAPE,R2
0,TOT_EMP,7942.239094,49531.715011,0.047036,0.984594
1,A_MEAN,1966.082034,4961.256766,0.034677,0.975658


In [10]:
predictions["ABS_EMP_ERROR"] = (predictions["ACTUAL_TOT_EMP"] - predictions["PREDICTED_TOT_EMP"]).abs()
predictions["EMP_PCT_ERROR"] = predictions["ABS_EMP_ERROR"] / predictions["ACTUAL_TOT_EMP"]
predictions["ABS_WAGE_ERROR"] = (predictions["ACTUAL_A_MEAN"] - predictions["PREDICTED_A_MEAN"]).abs()
predictions["WAGE_PCT_ERROR"] = predictions["ABS_WAGE_ERROR"] / predictions["ACTUAL_A_MEAN"]

largest_emp_errors = predictions.sort_values("ABS_EMP_ERROR", ascending=False).head(15)
largest_emp_errors[[
    "OCC_CODE",
    "OCC_TITLE",
    "ACTUAL_TOT_EMP",
    "PREDICTED_TOT_EMP",
    "EMP_PCT_ERROR",
    "ACTUAL_A_MEAN",
    "PREDICTED_A_MEAN",
    "WAGE_PCT_ERROR",
]]

,OCC_CODE,OCC_TITLE,ACTUAL_TOT_EMP,PREDICTED_TOT_EMP,EMP_PCT_ERROR,ACTUAL_A_MEAN,PREDICTED_A_MEAN,WAGE_PCT_ERROR
458,41-2031,Retail Salespersons,4612510,3.325597e+06,0.279005,26340,25394.808797,0.035884
319,29-1141,Registered Nurses,2745910,2.392695e+06,0.128633,71000,74478.955980,0.048999
447,39-9021,Personal Care Aides,1369230,1.199579e+06,0.123902,21790,23788.056560,0.091696
490,43-4051,Customer Service Representatives,2595990,2.438286e+06,0.060749,34560,34793.625499,0.006760
776,53-3032,Heavy and Tractor-Trailer Truck Drivers,1678280,1.544700e+06,0.079593,42500,41854.060202,0.015199
1,11-1021,General and Operations Managers,2145140,2.275201e+06,0.060630,119460,80453.152021,0.326526
664,51-2092,Team Assemblers,1115510,9.900077e+05,0.112507,31560,29941.690771,0.051277
401,35-3021,"Combined Food Preparation and Serving Workers,...",3216460,3.107243e+06,0.033956,19710,19955.814179,0.012472
515,43-6011,Executive Secretaries and Executive Administra...,666490,7.588426e+05,0.138566,55460,55784.350608,0.005848
354,31-1014,Nursing Assistants,1420570,1.335763e+06,0.059700,26820,26020.152431,0.029823


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(predictions["ACTUAL_TOT_EMP"], predictions["PREDICTED_TOT_EMP"], alpha=0.55)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_title("Employment: Actual vs Predicted")
axes[0].set_xlabel("Actual next-year employment")
axes[0].set_ylabel("Predicted next-year employment")

axes[1].scatter(predictions["ACTUAL_A_MEAN"], predictions["PREDICTED_A_MEAN"], alpha=0.55)
axes[1].set_title("Annual Mean Wage: Actual vs Predicted")
axes[1].set_xlabel("Actual next-year annual mean wage")
axes[1].set_ylabel("Predicted next-year annual mean wage")

plt.tight_layout()

In [12]:
# Train on all known one-year transitions, then forecast one year beyond the latest downloaded year.
final_model = model
final_model.fit(model_data[available_features], model_data[TARGET_COLUMNS])

latest_year = int(panel["YEAR"].max())
latest_features = panel[panel["YEAR"].eq(latest_year)].copy()
latest_predictions_log = final_model.predict(latest_features[available_features])

next_year_forecast = pd.DataFrame(
    {
        "FORECAST_FROM_YEAR": latest_year,
        "TARGET_YEAR": latest_year + 1,
        "OCC_CODE": latest_features["OCC_CODE"].to_numpy(),
        "OCC_TITLE": latest_features["OCC_TITLE"].to_numpy(),
        "CURRENT_TOT_EMP": latest_features["TOT_EMP"].to_numpy(),
        "FORECAST_TOT_EMP": np.expm1(latest_predictions_log[:, 0]),
        "CURRENT_A_MEAN": latest_features["A_MEAN"].to_numpy(),
        "FORECAST_A_MEAN": np.expm1(latest_predictions_log[:, 1]),
    }
)
next_year_forecast["FORECAST_EMP_CHANGE"] = next_year_forecast["FORECAST_TOT_EMP"] - next_year_forecast["CURRENT_TOT_EMP"]
next_year_forecast["FORECAST_EMP_GROWTH"] = next_year_forecast["FORECAST_EMP_CHANGE"] / next_year_forecast["CURRENT_TOT_EMP"]
next_year_forecast.sort_values("FORECAST_EMP_CHANGE", ascending=False).head(20)

,FORECAST_FROM_YEAR,TARGET_YEAR,OCC_CODE,OCC_TITLE,CURRENT_TOT_EMP,FORECAST_TOT_EMP,CURRENT_A_MEAN,FORECAST_A_MEAN,FORECAST_EMP_CHANGE,FORECAST_EMP_GROWTH
1,2015,2016,11-1021,General and Operations Managers,2145140,2.311807e+06,119460.0,82875.730869,166666.736459,0.077695
810,2015,2016,53-7062,"Laborers and Freight, Stock, and Material Move...",2487680,2.573252e+06,27840.0,29201.131335,85571.582243,0.034398
70,2015,2016,15-1132,"Software Developers, Applications",747730,8.313236e+05,102160.0,82884.185574,83593.635391,0.111797
53,2015,2016,13-2011,Accountants and Auditors,1226910,1.304268e+06,75280.0,71709.565646,77358.318228,0.063051
415,2015,2016,37-2011,"Janitors and Cleaners, Except Maids and Housek...",2146880,2.208866e+06,26180.0,26451.904110,61985.646441,0.028872
406,2015,2016,35-3022,"Counter Attendants, Cafeteria, Food Concession...",486650,5.322211e+05,20590.0,21629.244010,45571.061075,0.093642
400,2015,2016,35-2014,"Cooks, Restaurant",1150760,1.195987e+06,24430.0,24553.636501,45227.227825,0.039302
405,2015,2016,35-3021,"Combined Food Preparation and Serving Workers,...",3216460,3.260140e+06,19710.0,20271.012057,43679.529866,0.013580
52,2015,2016,13-1199,"Business Operations Specialists, All Other",926610,9.629572e+05,73480.0,71983.246161,36347.163046,0.039226
356,2015,2016,31-1011,Home Health Aides,820630,8.512147e+05,22870.0,23339.758027,30584.672415,0.037270


## Interpretation and Next Steps

The model is set up correctly for the data available so far, but the current download only reaches 2015. That means the validation test is limited to the 2014-to-2015 transition, and the forward forecast only projects 2016 from 2015.

For the full project proposal, the next step is to download and convert national occupation files for 2016-2025. Once those are available, rerun the notebook. It will automatically discover the additional files and produce a stronger time-aware evaluation.

Recommended final project workflow:

1. Train on 2010-2021.
2. Validate on 2022, 2023, 2024, and 2025 using walk-forward testing.
3. Compare Random Forest against simple baselines such as last-year carry-forward and linear trend.
4. Only try RNN/LSTM if you add much more frequent data, such as monthly job postings or quarterly labor-market indicators. Annual BLS files alone are too short for a reliable deep learning sequence model.
5. Report uncertainty clearly, especially for occupations affected by SOC classification changes or BLS suppression symbols.